# 🔌 Module 19 — Automatisation Python (API & scripts)
> **Bootcamp Data Analyst — From Zero to Hero** | Niveau Intermédiaire · Module 19

---

## 🎯 Ce que tu seras capable de faire à la fin de ce module

- Expliquer ce qu'est une API REST et à quoi elle sert pour un DA
- Faire une requête GET et parser une réponse JSON, en Python et en R
- Repérer un piège réel : une API peut renvoyer une erreur **dans le corps de la réponse**, avec un statut HTTP 200
- Écrire une fonction réutilisable, paramétrable, avec gestion d'erreurs
- Journaliser (logger) le résultat d'un script à chaque exécution
- Comprendre comment planifier l'exécution automatique d'un script
- Envelopper un script automatisé dans une mini app **Streamlit**

---

> ⏱️ **Durée estimée** : 2h30
> 🛠️ **Outils** : Python (`requests`, `streamlit`) · R (`httr`, `jsonlite`)
> 📌 **Prérequis** : Modules 14-15 (Niveau Intermédiaire)

---
## 1. Pourquoi automatiser

Jusqu'ici, chaque analyse que tu as faite était **ponctuelle** : tu charges un fichier, tu l'analyses, c'est fini. En poste, une bonne partie du travail d'un∙e DA consiste à répéter **la même extraction et la même analyse**, régulièrement — le taux de change du jour, les ventes de la veille, l'état des stocks. Refaire ça à la main chaque matin n'est ni fiable ni un bon usage de ton temps.

Ce module te donne les trois briques pour automatiser ce type de tâche :
1. **Récupérer la donnée automatiquement** depuis une API, sans intervention humaine
2. **Écrire un script réutilisable**, qui gère proprement ses erreurs
3. **Planifier son exécution**, et éventuellement l'exposer via une petite app

### Qu'est-ce qu'une API REST

Une **API** (Application Programming Interface) est un point d'accès qu'un service met à disposition pour que d'autres programmes puissent lui parler — sans interface graphique, juste des requêtes et des réponses. Une **API REST** répond à des requêtes HTTP (comme un navigateur web) et renvoie généralement les données au format **JSON**.

```
Ton script  ──── GET https://exemple.com/api/taux ────►  Serveur de l'API
Ton script  ◄─── réponse JSON {"USD": 574.48, ...} ─────  Serveur de l'API
```

C'est exactement ce que fait ton navigateur quand tu visites un site — sauf qu'ici, c'est ton script Python ou R qui joue le rôle du navigateur, et la réponse est pensée pour être lue par une machine plutôt que par un humain.

---
## 2. Faire une requête GET et lire le JSON

On utilise une API réelle, gratuite et sans clé : [open.er-api.com](https://open.er-api.com), qui donne les taux de change actuels entre devises — un scénario réaliste pour un∙e DA qui doit convertir des montants reçus en devises étrangères (USD, EUR...) en FCFA pour un reporting.

**Python (`requests`) :**
```python
import requests

reponse = requests.get("https://open.er-api.com/v6/latest/USD")
print(reponse.status_code)   # 200 = succès

data = reponse.json()        # convertit le JSON en dictionnaire Python
print(data["rates"]["XOF"])  # le taux USD -> FCFA
print(data["time_last_update_utc"])
```

**R (`httr` + `jsonlite`) :**
```r
library(httr)
library(jsonlite)

reponse <- GET("https://open.er-api.com/v6/latest/USD")
status_code(reponse)  # 200 = succès

data <- fromJSON(content(reponse, "text", encoding = "UTF-8"))
data$rates$XOF
data$time_last_update_utc
```

> 💡 Le taux USD → FCFA que tu obtiens **change chaque jour** (contrairement à EUR → FCFA, fixé par traité depuis la création du FCFA et donc toujours égal à 655,957). Relance ce code demain, tu obtiendras un chiffre différent pour USD — c'est le principe même de l'automatisation : le script reste identique, la donnée qu'il récupère évolue.

---
## 3. Un vrai piège — l'erreur qui ne dit pas son nom

Teste ton code avec un code de devise invalide :

```python
reponse = requests.get("https://open.er-api.com/v6/latest/ZZZ")
print(reponse.status_code)   # 200 !
print(reponse.json())        # {'result': 'error', 'error-type': 'unsupported-code'}
```

**Le statut HTTP est 200** (succès) alors que la requête a en réalité échoué — l'erreur est signalée **dans le corps de la réponse JSON** (`"result": "error"`), pas par le code de statut. Ce n'est pas un cas isolé : de nombreuses APIs fonctionnent ainsi.

> ⚠️ **Réflexe indispensable** : ne te fie jamais uniquement au code de statut HTTP. Vérifie toujours le contenu de la réponse avant de l'utiliser :

```python
def recuperer_taux(devise_source, devises_cibles):
    reponse = requests.get(f"https://open.er-api.com/v6/latest/{devise_source}", timeout=10)
    reponse.raise_for_status()          # leve une exception si le statut HTTP est une erreur (4xx/5xx)
    data = reponse.json()
    if data.get("result") != "success":  # verifie AUSSI le contenu, pas seulement le statut
        raise ValueError(f"L'API a renvoyé une erreur : {data.get('error-type')}")
    return {devise: data["rates"][devise] for devise in devises_cibles if devise in data["rates"]}
```

**R (même logique) :**
```r
recuperer_taux <- function(devise_source, devises_cibles) {
  reponse <- GET(paste0("https://open.er-api.com/v6/latest/", devise_source))
  stop_for_status(reponse)  # equivalent de raise_for_status()
  data <- fromJSON(content(reponse, "text", encoding = "UTF-8"))
  if (data$result != "success") {
    stop(paste("L'API a renvoyé une erreur :", data[["error-type"]]))
  }
  data$rates[devises_cibles]
}
```

---
## 4. Journaliser le résultat à chaque exécution

Un script d'automatisation utile garde une trace de ce qu'il a récupéré à chaque exécution — un simple fichier CSV auquel on **ajoute une ligne** à chaque appel suffit largement.

**Python :**
```python
import csv
import os
from datetime import datetime

def logger_taux(chemin_csv, devise_source="USD", devises_cibles=("XOF", "EUR")):
    taux = recuperer_taux(devise_source, devises_cibles)
    ligne = {"horodatage": datetime.now().isoformat(), "devise_source": devise_source, **taux}

    fichier_existe = os.path.exists(chemin_csv)
    with open(chemin_csv, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(ligne.keys()))
        if not fichier_existe:
            writer.writeheader()
        writer.writerow(ligne)
    return ligne

logger_taux("historique_taux.csv")
```

Exécute ce script deux jours de suite : `historique_taux.csv` accumule une ligne à chaque fois, avec un horodatage — tu commences à construire un historique que tu pourras analyser plus tard (avec les outils du Module 15).

**R (équivalent avec `write.table` en mode ajout) :**
```r
logger_taux <- function(chemin_csv, devise_source = "USD", devises_cibles = c("XOF", "EUR")) {
  taux <- recuperer_taux(devise_source, devises_cibles)
  ligne <- data.frame(horodatage = Sys.time(), devise_source = devise_source, t(taux))

  fichier_existe <- file.exists(chemin_csv)
  write.table(ligne, chemin_csv, sep = ",", append = fichier_existe,
              col.names = !fichier_existe, row.names = FALSE)
}
```

---
## 5. Planifier l'exécution automatique

Un script qu'il faut lancer soi-même chaque jour n'est qu'à moitié automatisé. Pour une vraie automatisation, il faut le **planifier** :

| Environnement | Comment planifier |
|---|---|
| **Windows** | Planificateur de tâches (Task Scheduler) — exécute ton script `.py` à une heure fixe, tous les jours |
| **Mac/Linux** | `cron` — une ligne dans la crontab (`0 8 * * * python3 /chemin/script.py` = tous les jours à 8h) |
| **Dans le script Python lui-même** | La librairie `schedule` : `schedule.every().day.at("08:00").do(logger_taux, ...)`, puis une boucle qui vérifie régulièrement s'il est temps d'exécuter |
| **Cloud** | Un job planifié hébergé (GitHub Actions avec un déclencheur `cron`, par exemple) — ton script tourne même si ton ordinateur est éteint |

> 💡 Pour un script qui doit tourner tous les jours sans que ton ordinateur reste allumé, un job planifié dans le cloud (GitHub Actions, par exemple — tu as déjà un dépôt GitHub depuis le Module 13) est l'option la plus robuste.

---
## 6. Envelopper le script dans une mini app Streamlit

**Streamlit** transforme un script Python en application web interactive, partageable, en quelques lignes — sans rien connaître au développement web.

### Installer

```bash
pip install streamlit
```

### Une première app

Crée un fichier `app.py` :

```python
import streamlit as st
import pandas as pd

st.title("💱 Taux de change FCFA")

devise_source = st.selectbox("Devise source", ["USD", "EUR", "GBP"])

if st.button("Récupérer le taux actuel"):
    taux = recuperer_taux(devise_source, ("XOF",))
    st.metric(label=f"{devise_source} → XOF", value=f"{taux['XOF']:.2f} FCFA")
```

### Lancer l'app

```bash
streamlit run app.py
```

Ton navigateur s'ouvre automatiquement sur une petite application web : un menu déroulant pour choisir la devise, un bouton qui déclenche l'appel API en direct, et le résultat affiché immédiatement. C'est le même script d'automatisation que tu as écrit en section 2-3 — Streamlit ne fait qu'ajouter une couche de présentation par-dessus.

> 🔗 **Pont vers le Mini-Projet C** : c'est exactement le schéma que tu vas construire pour le pipeline complet — API (ce module) → nettoyage (Modules 14-15) → mini-app Streamlit (ce module).

---
## ✅ Résumé du module

| Concept | Ce qu'il faut retenir |
|---|---|
| **API REST** | Point d'accès qui répond à des requêtes HTTP et renvoie des données, généralement en JSON |
| **`requests.get()` / `GET()` (httr)** | Fait une requête GET vers une API |
| **`.json()` / `fromJSON(content(...))`** | Convertit la réponse en dictionnaire/liste exploitable |
| **Statut HTTP 200 ≠ succès garanti** | Certaines APIs renvoient une erreur dans le corps de la réponse JSON, avec un statut 200 — toujours vérifier le contenu, pas seulement le statut |
| **`raise_for_status()` / `stop_for_status()`** | Lève une exception si le statut HTTP est une erreur (4xx/5xx) — ne couvre pas les erreurs "silencieuses" ci-dessus |
| **Journaliser (logger)** | Ajouter une ligne horodatée à un fichier à chaque exécution, pour construire un historique |
| **Planifier** | Task Scheduler (Windows), `cron` (Mac/Linux), librairie `schedule`, ou un job cloud (GitHub Actions) |
| **Streamlit** | Transforme un script Python en app web interactive et partageable, en quelques lignes |

---

## 🧠 Quiz — Vérifie ta compréhension

---

**Q1.** Une API te renvoie un statut HTTP 200, mais le corps de la réponse contient `{"result": "error", ...}`. Que dois-tu en conclure ?
- a) La requête a réussi, tu peux utiliser les données sans vérification
- b) `raise_for_status()` aurait dû lever une exception automatiquement
- c) Il faut aussi vérifier le contenu de la réponse — certaines APIs signalent une erreur dans le corps, pas seulement via le statut HTTP

<details>
<summary>👉 Voir la réponse</summary>

✅ **c)** — `raise_for_status()` (b) ne réagit qu'aux codes de statut HTTP d'erreur (4xx/5xx) ; un statut 200 avec une erreur "cachée" dans le JSON ne déclenchera jamais cette exception. Il faut toujours vérifier explicitement le contenu de la réponse.
</details>

---

**Q2.** Pourquoi journaliser (logger) chaque exécution d'un script d'automatisation dans un fichier, plutôt que de juste afficher le résultat à l'écran ?
- a) Ça rend le script plus rapide
- b) Ça construit un historique exploitable plus tard, même si personne n'a regardé l'écran au moment de l'exécution
- c) C'est obligatoire pour que Python fonctionne

<details>
<summary>👉 Voir la réponse</summary>

✅ **b)** — Un script planifié (cron, Task Scheduler) tourne souvent sans personne devant l'écran. Journaliser dans un fichier garantit que le résultat de chaque exécution reste disponible pour une analyse ultérieure.
</details>

---

**Q3.** Qu'est-ce que Streamlit ajoute par rapport à ton script Python d'origine ?
- a) Il rend les appels API plus rapides
- b) Une couche de présentation web interactive, sans changer la logique du script
- c) Il remplace la nécessité d'écrire du code Python

<details>
<summary>👉 Voir la réponse</summary>

✅ **b)** — Streamlit enveloppe ton code existant (la fonction `recuperer_taux`, par exemple) dans une interface web cliquable — la logique métier ne change pas, seule la présentation devient interactive et partageable.
</details>

---

## ➡️ Module suivant

Tu sais maintenant automatiser une extraction de données et l'exposer via une mini app. Dans le **Module 20**, on passe au storytelling — comment présenter tes résultats pour convaincre un∙e décideur∙se.

> **→ Module 20 — Storytelling + IA**